# Auditoría temporal de la selección de variables

## 1. Preparación del conjunto de auditoría

El objetivo de este análisis es comprobar si las decisiones supervisadas adoptadas durante la selección exploratoria de variables se mantienen cuando se reproducen utilizando exclusivamente la información disponible en el conjunto de entrenamiento de cada partición temporal.

Esta auditoría no pretende realizar una nueva selección optimizada de predictores ni modificar las decisiones ya adoptadas en función de los resultados obtenidos. Su finalidad es evaluar la estabilidad temporal de la evidencia que respaldó dichas decisiones.

Se parte del conjunto previo a la selección final de predictores tradicionales, formado por los 47 candidatos disponibles antes del partido. La temporada 2024 permanece completamente excluida del análisis.

Las dos observaciones recíprocas correspondientes a un mismo encuentro se mantienen vinculadas mediante `_id_partido`.

### Procedencia del conjunto de preselección tradicional

El archivo `df_auditoria_preseleccion_tradicional.csv` no constituye una fuente de datos externa ni un conjunto independiente del pipeline. Se conserva como un **snapshot intermedio congelado** de la fase de preparación y selección de predictores tradicionales realizada en `02_transformacion_y_preparacion_predictores_tradicionales.ipynb`.

El snapshot corresponde al estado del dataset jugador–partido después de:

- construir la estructura recíproca jugador–partido;
- aplicar el tratamiento y la codificación de las variables categóricas;
- transformar la condición de cabeza de serie;
- obtener los **47 predictores numéricos candidatos**;
- restringir el análisis al conjunto de desarrollo mediante `es_desarrollo_modelo == 1`.

Por tanto, contiene exclusivamente las temporadas de desarrollo **2009–2019 y 2022–2023**, manteniendo fuera los años buffer 2008 y 2021 y la temporada 2024 reservada para la evaluación temporal externa.

El archivo fue conservado **antes de aplicar la selección final de predictores tradicionales**, es decir, antes de eliminar las variables redundantes o categorías finalmente descartadas y antes de sustituir las variables individuales de ranking, edad y altura del jugador y del rival por sus correspondientes diferencias jugador–rival. Su finalidad es permitir reconstruir retrospectivamente, dentro de cada fold temporal, la evidencia estadística disponible en el momento de la selección original.

El snapshot contiene **76 294 observaciones jugador–partido, correspondientes a 38 147 encuentros**. Esta cifra es dos observaciones superior a la utilizada posteriormente en los datasets de carga porque el snapshot se generó antes de la comprobación de calidad que eliminó un único partido sin volumen competitivo observable —dos filas recíprocas—. Esta diferencia es, por tanto, deliberada y refleja dos momentos distintos del pipeline.

Aunque el archivo conserva algunas columnas auxiliares del estado histórico del dataset, estas no se consideran automáticamente predictores. En particular, `victoria` se excluye explícitamente del conjunto de variables candidatas y la auditoría supervisada utiliza únicamente `porcentaje_juegos_ganados` como variable objetivo.

Este conjunto se utiliza exclusivamente para la **auditoría retrospectiva de estabilidad temporal** y no interviene en el entrenamiento, la selección definitiva del modelo ni la evaluación externa de 2024.

In [1]:
import pandas as pd
import numpy as np

df_auditoria = pd.read_csv(
    "df_auditoria_preseleccion_tradicional.csv",
    low_memory=False
)

df_auditoria["fecha_torneo"] = pd.to_datetime(
    df_auditoria["fecha_torneo"],
    errors="raise"
)

print("Dimensión:", df_auditoria.shape)
print("Partidos:", df_auditoria["_id_partido"].nunique())
print("Años:", sorted(df_auditoria["match_year"].unique()))

assert len(df_auditoria) == 76294
assert df_auditoria["_id_partido"].nunique() == 38147
assert (
    df_auditoria
    .groupby("_id_partido")
    .size()
    .eq(2)
    .all()
)
assert 2024 not in df_auditoria["match_year"].unique()

print("Estructura del conjunto de auditoría validada correctamente.")

Dimensión: (76294, 57)
Partidos: 38147
Años: [np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2022), np.int64(2023)]
Estructura del conjunto de auditoría validada correctamente.


## 2. Particiones temporales de la auditoría

La auditoría reproduce las mismas cinco particiones temporales utilizadas durante el desarrollo del modelado.

En cada fold, los análisis supervisados de selección se realizarán exclusivamente con las temporadas pertenecientes al conjunto de entrenamiento. El año utilizado como validación permanecerá completamente excluido de dichas decisiones.

Las particiones son:

- Fold 1: entrenamiento 2009–2016 y validación 2017.
- Fold 2: entrenamiento 2009–2017 y validación 2018.
- Fold 3: entrenamiento 2009–2018 y validación 2019.
- Fold 4: entrenamiento 2009–2019 y validación 2022.
- Fold 5: entrenamiento 2009–2019 y 2022, y validación 2023.

El objetivo no es volver a optimizar el procedimiento, sino comprobar retrospectivamente si la evidencia utilizada para seleccionar los predictores permanece estable cuando solo se utiliza la información temporalmente disponible en cada conjunto de entrenamiento.

In [2]:
folds_auditoria = [
    {
        "fold": 1,
        "anios_train": list(range(2009, 2017)),
        "anio_validacion": 2017
    },
    {
        "fold": 2,
        "anios_train": list(range(2009, 2018)),
        "anio_validacion": 2018
    },
    {
        "fold": 3,
        "anios_train": list(range(2009, 2019)),
        "anio_validacion": 2019
    },
    {
        "fold": 4,
        "anios_train": list(range(2009, 2020)),
        "anio_validacion": 2022
    },
    {
        "fold": 5,
        "anios_train": list(range(2009, 2020)) + [2022],
        "anio_validacion": 2023
    }
]

resumen_folds_auditoria = []

for fold in folds_auditoria:

    train = df_auditoria[
        df_auditoria["match_year"].isin(fold["anios_train"])
    ]

    validacion = df_auditoria[
        df_auditoria["match_year"].eq(fold["anio_validacion"])
    ]

    assert len(train) > 0
    assert len(validacion) > 0

    assert (
        train["fecha_torneo"].max()
        <
        validacion["fecha_torneo"].min()
    )

    assert set(train["_id_partido"]).isdisjoint(
        set(validacion["_id_partido"])
    )

    resumen_folds_auditoria.append(
        {
            "fold": fold["fold"],
            "anios_train": str(fold["anios_train"]),
            "validacion": fold["anio_validacion"],
            "observaciones_train": len(train),
            "partidos_train": train["_id_partido"].nunique(),
            "observaciones_validacion": len(validacion),
            "partidos_validacion": validacion["_id_partido"].nunique()
        }
    )

resumen_folds_auditoria = pd.DataFrame(
    resumen_folds_auditoria
)

display(resumen_folds_auditoria)

print("Particiones temporales de la auditoría: CORRECTAS")

,fold,anios_train,validacion,observaciones_train,partidos_train,observaciones_validacion,partidos_validacion
0,1,"[2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016]",2017,47444,23722,5784,2892
1,2,"[2009, 2010, 2011, 2012, 2013, 2014, 2015, 201...",2018,53228,26614,5764,2882
2,3,"[2009, 2010, 2011, 2012, 2013, 2014, 2015, 201...",2019,58992,29496,5570,2785
3,4,"[2009, 2010, 2011, 2012, 2013, 2014, 2015, 201...",2022,64562,32281,5800,2900
4,5,"[2009, 2010, 2011, 2012, 2013, 2014, 2015, 201...",2023,70362,35181,5932,2966


Particiones temporales de la auditoría: CORRECTAS


## 3. Estabilidad temporal de la evidencia supervisada

La selección original de predictores tradicionales no se realizó mediante un algoritmo automático ni mediante un umbral único. No obstante, parte de la evidencia utilizada para respaldar las decisiones procedía de medidas supervisadas calculadas respecto al rendimiento.

Para evaluar si dicha evidencia dependía de haber utilizado el conjunto completo de desarrollo, se reproducen dentro del entrenamiento de cada fold temporal:

- correlación de Pearson;
- correlación de Spearman;
- correlación de Kendall;
- información mutua.

El análisis se realiza  respecto a `porcentaje_juegos_ganados`.

No se utilizarán estas medidas para realizar una nueva optimización de variables. El objetivo es comprobar si la estructura de asociaciones observada originalmente permanece estable cuando únicamente se utiliza la información disponible en cada periodo de entrenamiento.

In [3]:
from sklearn.feature_selection import mutual_info_regression

# ============================================================
# PREDICTORES CANDIDATOS
# ============================================================

columnas_no_predictoras = [
    "identificador_torneo",
    "numero_partido",
    "fecha_torneo",
    "match_year",
    "jugador_id",
    "rival_id",
    "es_desarrollo_modelo",
    "porcentaje_juegos_ganados",
    "victoria",
    "_id_partido"
]

predictores_candidatos = [
    col
    for col in df_auditoria.columns
    if col not in columnas_no_predictoras
]

assert len(predictores_candidatos) == 47


# ============================================================
# VARIABLES DISCRETAS
# Mismo criterio utilizado en el análisis original de MI
# ============================================================

predictores_discretos = [
    variable
    for variable in predictores_candidatos
    if (
        variable in [
            "tamano_cuadro",
            "numero_maximo_sets",
            "ronda_encoded",
            "es_round_robin",
            "cabeza_serie",
            "cabeza_serie_rival"
        ]
        or variable.startswith("superficie_")
        or variable.startswith("nivel_torneo_")
        or variable.startswith("mano_dominante_")
        or variable.startswith("tipo_entrada_")
    )
]

mascara_discretos = [
    variable in predictores_discretos
    for variable in predictores_candidatos
]


# ============================================================
# CÁLCULO POR FOLD
# ============================================================

resultados_supervisados = []
resumen_muestras_mi = []

for fold in folds_auditoria:

    datos_train = df_auditoria[
        df_auditoria["match_year"].isin(
            fold["anios_train"]
        )
    ].copy()

    # --------------------------------------------------------
    # Correlaciones
    # pandas utiliza los casos disponibles para cada par.
    # --------------------------------------------------------

    columnas_correlacion = (
        predictores_candidatos
        + ["porcentaje_juegos_ganados"]
    )

    pearson = datos_train[
        columnas_correlacion
    ].corr(method="pearson")

    spearman = datos_train[
        columnas_correlacion
    ].corr(method="spearman")

    kendall = datos_train[
        columnas_correlacion
    ].corr(method="kendall")

    # --------------------------------------------------------
    # Información mutua
    # Igual que en el análisis original: casos completos
    # en todos los predictores considerados.
    # --------------------------------------------------------

    datos_mi = (
        datos_train[
            predictores_candidatos
            + ["porcentaje_juegos_ganados"]
        ]
        .dropna()
        .copy()
    )

    X_mi = datos_mi[predictores_candidatos]

    mi_porcentaje = mutual_info_regression(
        X=X_mi,
        y=datos_mi["porcentaje_juegos_ganados"],
        discrete_features=mascara_discretos,
        random_state=42
    )

   

    # --------------------------------------------------------
    # Guardamos resultados predictor a predictor
    # --------------------------------------------------------

    for i, variable in enumerate(predictores_candidatos):

        resultados_supervisados.append(
            {
                "fold": fold["fold"],
                "variable": variable,

                "pearson_porcentaje":
                    pearson.loc[
                        variable,
                        "porcentaje_juegos_ganados"
                    ],

                "spearman_porcentaje":
                    spearman.loc[
                        variable,
                        "porcentaje_juegos_ganados"
                    ],

                "kendall_porcentaje":
                    kendall.loc[
                        variable,
                        "porcentaje_juegos_ganados"
                    ],

                "mi_porcentaje":
                    mi_porcentaje[i],

            }
        )

    resumen_muestras_mi.append(
        {
            "fold": fold["fold"],
            "observaciones_train": len(datos_train),
            "observaciones_completas_mi": len(datos_mi),
            "porcentaje_completo":
                len(datos_mi) / len(datos_train) * 100
        }
    )


resultados_supervisados = pd.DataFrame(
    resultados_supervisados
)

resumen_muestras_mi = pd.DataFrame(
    resumen_muestras_mi
)

display(resumen_muestras_mi.round(2))

print(
    "Predictores analizados:",
    resultados_supervisados["variable"].nunique()
)

print(
    "Folds analizados:",
    resultados_supervisados["fold"].nunique()
)

assert resultados_supervisados["variable"].nunique() == 47
assert resultados_supervisados["fold"].nunique() == 5


print("Evidencia supervisada calculada correctamente.")

,fold,observaciones_train,observaciones_completas_mi,porcentaje_completo
0,1,47444,45014,94.88
1,2,53228,50618,95.10
2,3,58992,56238,95.33
3,4,64562,61698,95.56
4,5,70362,67426,95.83


Predictores analizados: 47
Folds analizados: 5
Evidencia supervisada calculada correctamente.


## 4. Comparación con la evidencia obtenida sobre el desarrollo completo

Para evaluar la estabilidad temporal de los análisis supervisados, se utiliza como referencia la evidencia obtenida al considerar el conjunto completo de desarrollo.

Las mismas medidas se recalculan sobre 2009–2019 y 2022–2023 utilizando exactamente los mismos predictores y criterios empleados dentro de los folds.

Posteriormente, para cada partición temporal se compara la ordenación de los 47 predictores con la obtenida sobre el desarrollo completo.

En las correlaciones de Pearson, Spearman y Kendall se considera la magnitud absoluta de la asociación, ya que una relación negativa intensa puede contener tanta información como una relación positiva de magnitud equivalente. En la información mutua se utilizan directamente sus valores, al tratarse de una medida no negativa.

La similitud entre ordenaciones se cuantifica mediante la correlación de rangos de Spearman. Valores próximos a uno indican que la estructura relativa de asociaciones permanece estable entre periodos, mientras que valores reducidos señalarían una mayor sensibilidad temporal.

Este análisis no establece umbrales automáticos de selección. Su finalidad es comprobar si la evidencia supervisada que contribuyó a las decisiones originales presenta una estructura temporal estable.

In [4]:
# ============================================================
# REFERENCIA: DESARROLLO COMPLETO
# ============================================================

columnas_correlacion = (
    predictores_candidatos
    + ["porcentaje_juegos_ganados"]
)

pearson_global = df_auditoria[
    columnas_correlacion
].corr(method="pearson")

spearman_global = df_auditoria[
    columnas_correlacion
].corr(method="spearman")

kendall_global = df_auditoria[
    columnas_correlacion
].corr(method="kendall")


# Información mutua sobre casos completos,
# reproduciendo el criterio del análisis original.
datos_mi_global = (
    df_auditoria[
        predictores_candidatos
        + ["porcentaje_juegos_ganados"]
    ]
    .dropna()
    .copy()
)

mi_global = mutual_info_regression(
    X=datos_mi_global[predictores_candidatos],
    y=datos_mi_global["porcentaje_juegos_ganados"],
    discrete_features=mascara_discretos,
    random_state=42
)


referencia_global = pd.DataFrame({
    "variable": predictores_candidatos,

    "pearson_global": [
        pearson_global.loc[
            variable,
            "porcentaje_juegos_ganados"
        ]
        for variable in predictores_candidatos
    ],

    "spearman_global": [
        spearman_global.loc[
            variable,
            "porcentaje_juegos_ganados"
        ]
        for variable in predictores_candidatos
    ],

    "kendall_global": [
        kendall_global.loc[
            variable,
            "porcentaje_juegos_ganados"
        ]
        for variable in predictores_candidatos
    ],

    "mi_global": mi_global
})


# ============================================================
# ESTABILIDAD DE CADA FOLD RESPECTO AL DESARROLLO COMPLETO
# ============================================================

estabilidad_folds = []

for numero_fold in sorted(
    resultados_supervisados["fold"].unique()
):

    resultados_fold = (
        resultados_supervisados[
            resultados_supervisados["fold"].eq(
                numero_fold
            )
        ]
        .merge(
            referencia_global,
            on="variable",
            how="inner"
        )
    )

    assert len(resultados_fold) == len(
        predictores_candidatos
    )

    estabilidad_folds.append({
        "fold": numero_fold,

        "pearson":
            resultados_fold[
                "pearson_porcentaje"
            ].abs().corr(
                resultados_fold[
                    "pearson_global"
                ].abs(),
                method="spearman"
            ),

        "spearman":
            resultados_fold[
                "spearman_porcentaje"
            ].abs().corr(
                resultados_fold[
                    "spearman_global"
                ].abs(),
                method="spearman"
            ),

        "kendall":
            resultados_fold[
                "kendall_porcentaje"
            ].abs().corr(
                resultados_fold[
                    "kendall_global"
                ].abs(),
                method="spearman"
            ),

        "informacion_mutua":
            resultados_fold[
                "mi_porcentaje"
            ].corr(
                resultados_fold[
                    "mi_global"
                ],
                method="spearman"
            )
    })


estabilidad_folds = pd.DataFrame(
    estabilidad_folds
)

print(
    "Observaciones completas MI "
    "en desarrollo completo:",
    len(datos_mi_global)
)

display(
    estabilidad_folds.round(3)
)

Observaciones completas MI en desarrollo completo: 73242


,fold,pearson,spearman,kendall,informacion_mutua
0,1,0.976,0.995,0.996,0.910
1,2,0.980,0.994,0.990,0.896
2,3,0.987,0.996,0.994,0.972
3,4,0.989,0.996,0.994,0.918
4,5,0.993,0.996,0.997,0.930


### Resultado de la estabilidad global

La estructura de la evidencia supervisada presentó una elevada estabilidad temporal. Las ordenaciones de los 47 predictores obtenidas dentro de los conjuntos de entrenamiento mostraron una concordancia muy alta con la obtenida sobre el desarrollo completo.

Las correlaciones de rangos asociadas a Pearson, Spearman y Kendall se mantuvieron próximas a uno en las cinco particiones. La información mutua presentó una variabilidad algo mayor, aunque conservó igualmente una elevada concordancia entre periodos.

Estos resultados indican que la estructura general de asociación entre los predictores tradicionales y el rendimiento ya estaba presente en los periodos de entrenamiento anteriores y no parece depender fuertemente de las temporadas más recientes del conjunto de desarrollo.

No obstante, dado el carácter expansivo de las particiones y el solapamiento existente entre sus muestras, estos resultados se interpretan como evidencia descriptiva de estabilidad y no como una prueba de independencia completa. Por este motivo, la auditoría continúa examinando específicamente las decisiones de selección adoptadas sobre las variables individuales.

### Comprobación de estabilidad sobre periodos no solapados

La comparación anterior mostró una elevada estabilidad entre la evidencia obtenida en los conjuntos de entrenamiento y la correspondiente al desarrollo completo. No obstante, dado que las particiones expansivas comparten una parte importante de sus observaciones con el conjunto de referencia, dicha concordancia puede verse parcialmente favorecida por el solapamiento de las muestras.

Como comprobación adicional, se compara la estructura de asociaciones obtenida en el entrenamiento de cada fold con la observada en su correspondiente periodo de validación, que no comparte partidos con el conjunto de entrenamiento.

La comparación se realiza mediante la correlación de rangos de Spearman entre las magnitudes de las asociaciones predictor-rendimiento obtenidas en ambos periodos.

Este análisis tiene una finalidad exclusivamente descriptiva de estabilidad temporal. Los resultados de los periodos de validación no se utilizarán para modificar la selección de variables.

In [5]:
estabilidad_fuera_muestra = []

for fold in folds_auditoria:

    train = df_auditoria[
        df_auditoria["match_year"].isin(
            fold["anios_train"]
        )
    ].copy()

    validacion = df_auditoria[
        df_auditoria["match_year"].eq(
            fold["anio_validacion"]
        )
    ].copy()

    columnas = (
        predictores_candidatos
        + ["porcentaje_juegos_ganados"]
    )

    spearman_train = (
        train[columnas]
        .corr(method="spearman")[
            "porcentaje_juegos_ganados"
        ]
        .drop("porcentaje_juegos_ganados")
    )

    spearman_validacion = (
        validacion[columnas]
        .corr(method="spearman")[
            "porcentaje_juegos_ganados"
        ]
        .drop("porcentaje_juegos_ganados")
    )

    comparacion = pd.concat(
        [
            spearman_train.abs(),
            spearman_validacion.abs()
        ],
        axis=1,
        keys=[
            "train",
            "validacion"
        ]
    ).dropna()

    estabilidad_fuera_muestra.append(
        {
            "fold": fold["fold"],
            "anio_validacion":
                fold["anio_validacion"],
            "predictores_comparables":
                len(comparacion),
            "correlacion_rangos":
                comparacion[
                    "train"
                ].corr(
                    comparacion[
                        "validacion"
                    ],
                    method="spearman"
                )
        }
    )

estabilidad_fuera_muestra = pd.DataFrame(
    estabilidad_fuera_muestra
)

display(
    estabilidad_fuera_muestra.round(3)
)

,fold,anio_validacion,predictores_comparables,correlacion_rangos
0,1,2017,41,0.981
1,2,2018,42,0.983
2,3,2019,42,0.980
3,4,2022,42,0.964
4,5,2023,42,0.977


### Predictores no comparables en periodos concretos

En algunos folds no pueden calcularse asociaciones para los 47 predictores candidatos. Esto ocurre cuando alguna variable carece de variabilidad suficiente en el conjunto de entrenamiento o en el periodo de validación, o cuando la correlación resultante no está definida.

Para que el diagnóstico sea coherente con la comparación utilizada en la auditoría, se identifican directamente los predictores cuya correlación de Spearman con `porcentaje_juegos_ganados` no puede calcularse en entrenamiento, en validación o en ambos conjuntos.

Estas variables se excluyen únicamente de la comparación de rangos correspondiente a ese fold; no se utiliza esta información para modificar retrospectivamente la selección original.

In [6]:
predictores_no_comparables = []

for fold in folds_auditoria:

    train = df_auditoria[
        df_auditoria["match_year"].isin(
            fold["anios_train"]
        )
    ].copy()

    validacion = df_auditoria[
        df_auditoria["match_year"].eq(
            fold["anio_validacion"]
        )
    ].copy()

    columnas = (
        predictores_candidatos
        + ["porcentaje_juegos_ganados"]
    )

    spearman_train = (
        train[columnas]
        .corr(method="spearman")[
            "porcentaje_juegos_ganados"
        ]
        .drop("porcentaje_juegos_ganados")
    )

    spearman_validacion = (
        validacion[columnas]
        .corr(method="spearman")[
            "porcentaje_juegos_ganados"
        ]
        .drop("porcentaje_juegos_ganados")
    )

    no_comparables = [
        variable
        for variable in predictores_candidatos
        if (
            pd.isna(spearman_train[variable])
            or pd.isna(spearman_validacion[variable])
        )
    ]

    solo_train = [
        variable
        for variable in predictores_candidatos
        if (
            pd.isna(spearman_train[variable])
            and not pd.isna(spearman_validacion[variable])
        )
    ]

    solo_validacion = [
        variable
        for variable in predictores_candidatos
        if (
            not pd.isna(spearman_train[variable])
            and pd.isna(spearman_validacion[variable])
        )
    ]

    ambos = [
        variable
        for variable in predictores_candidatos
        if (
            pd.isna(spearman_train[variable])
            and pd.isna(spearman_validacion[variable])
        )
    ]

    predictores_no_comparables.append(
        {
            "fold": fold["fold"],
            "anio_validacion": fold["anio_validacion"],
            "n_no_comparables": len(no_comparables),
            "variables": no_comparables,
            "solo_train": solo_train,
            "solo_validacion": solo_validacion,
            "ambos": ambos
        }
    )

predictores_no_comparables = pd.DataFrame(
    predictores_no_comparables
)

comparables_esperados = (
    len(predictores_candidatos)
    - predictores_no_comparables["n_no_comparables"]
)

assert comparables_esperados.tolist() == (
    estabilidad_fuera_muestra[
        "predictores_comparables"
    ].tolist()
)

display(predictores_no_comparables)

,fold,anio_validacion,n_no_comparables,variables,solo_train,solo_validacion,ambos
0,1,2017,6,"[superficie_UNKNOWN, nivel_torneo_Olympics, ti...","[tipo_entrada_ALT, tipo_entrada_rival_ALT]","[tipo_entrada_OTROS, tipo_entrada_rival_OTROS]","[superficie_UNKNOWN, nivel_torneo_Olympics]"
1,2,2018,5,"[superficie_Carpet, superficie_UNKNOWN, nivel_...",[],"[superficie_Carpet, tipo_entrada_OTROS, tipo_e...","[superficie_UNKNOWN, nivel_torneo_Olympics]"
2,3,2019,5,"[superficie_Carpet, superficie_UNKNOWN, nivel_...",[],"[superficie_Carpet, tipo_entrada_OTROS, tipo_e...","[superficie_UNKNOWN, nivel_torneo_Olympics]"
3,4,2022,5,"[superficie_Carpet, superficie_UNKNOWN, nivel_...",[],"[superficie_Carpet, tipo_entrada_OTROS, tipo_e...","[superficie_UNKNOWN, nivel_torneo_Olympics]"
4,5,2023,5,"[superficie_Carpet, superficie_UNKNOWN, nivel_...",[superficie_UNKNOWN],"[superficie_Carpet, tipo_entrada_OTROS, tipo_e...",[nivel_torneo_Olympics]


### Estabilidad temporal de la redundancia entre ranking y puntos ATP

Una de las principales decisiones de depuración del modelo base consistió en eliminar los puntos ATP del jugador y del rival, conservando únicamente sus posiciones de ranking.

La decisión original se apoyó en la elevada redundancia observada entre ambas representaciones del nivel competitivo. Para comprobar que esta relación no dependía del uso del conjunto completo de desarrollo, se recalcula la correlación de Spearman entre ranking y puntos ATP utilizando exclusivamente el conjunto de entrenamiento de cada fold temporal.

Una relación negativa próxima a -1 indicaría que ambas variables continúan representando prácticamente la misma dimensión competitiva en los distintos periodos.

In [7]:
redundancia_ranking = []

for fold in folds_auditoria:

    train = df_auditoria[
        df_auditoria["match_year"].isin(
            fold["anios_train"]
        )
    ]

    correlacion_jugador = (
        train[
            ["ranking", "puntos_ranking"]
        ]
        .corr(method="spearman")
        .loc["ranking", "puntos_ranking"]
    )

    correlacion_rival = (
        train[
            ["ranking_rival", "puntos_ranking_rival"]
        ]
        .corr(method="spearman")
        .loc[
            "ranking_rival",
            "puntos_ranking_rival"
        ]
    )

    redundancia_ranking.append(
        {
            "fold": fold["fold"],
            "ranking_puntos_jugador":
                correlacion_jugador,
            "ranking_puntos_rival":
                correlacion_rival
        }
    )

redundancia_ranking = pd.DataFrame(
    redundancia_ranking
)

display(
    redundancia_ranking.round(4)
)

,fold,ranking_puntos_jugador,ranking_puntos_rival
0,1,-0.9876,-0.9876
1,2,-0.9887,-0.9887
2,3,-0.9893,-0.9893
3,4,-0.9896,-0.9896
4,5,-0.9890,-0.9890


### Estabilidad temporal de las variables diferenciales jugador–rival

Tras la depuración inicial se construyeron diferencias entre las características del jugador y de su rival para ranking, edad y altura.

Estas variables permiten representar directamente la situación relativa de ambos participantes y evitan incorporar simultáneamente las dos medidas individuales y su diferencia, lo que introduciría una dependencia lineal exacta.

Como comprobación temporal, se compara dentro del entrenamiento de cada fold la asociación de las variables individuales y de sus correspondientes diferencias con `porcentaje_juegos_ganados`.

El objetivo no es realizar una nueva selección, sino comprobar si la utilidad de la representación diferencial observada originalmente se mantiene utilizando exclusivamente la información disponible en cada periodo de entrenamiento.

In [8]:
resultados_diferencias = []

for fold in folds_auditoria:

    train = df_auditoria[
        df_auditoria["match_year"].isin(
            fold["anios_train"]
        )
    ].copy()

    # Construcción de las mismas diferencias utilizadas
    # posteriormente en el modelo base.
    train["diff_ranking"] = (
        train["ranking_rival"]
        - train["ranking"]
    )

    train["diff_edad"] = (
        train["edad"]
        - train["edad_rival"]
    )

    train["diff_altura"] = (
        train["altura"]
        - train["altura_rival"]
    )

    variables = [
        "ranking",
        "ranking_rival",
        "diff_ranking",
        "edad",
        "edad_rival",
        "diff_edad",
        "altura",
        "altura_rival",
        "diff_altura",
        "porcentaje_juegos_ganados"
    ]

    correlaciones = (
        train[variables]
        .corr(method="spearman")[
            "porcentaje_juegos_ganados"
        ]
    )

    resultados_diferencias.append(
        {
            "fold": fold["fold"],

            "ranking_jugador":
                abs(correlaciones["ranking"]),

            "ranking_rival":
                abs(correlaciones["ranking_rival"]),

            "diff_ranking":
                abs(correlaciones["diff_ranking"]),

            "edad_jugador":
                abs(correlaciones["edad"]),

            "edad_rival":
                abs(correlaciones["edad_rival"]),

            "diff_edad":
                abs(correlaciones["diff_edad"]),

            "altura_jugador":
                abs(correlaciones["altura"]),

            "altura_rival":
                abs(correlaciones["altura_rival"]),

            "diff_altura":
                abs(correlaciones["diff_altura"])
        }
    )

resultados_diferencias = pd.DataFrame(
    resultados_diferencias
)

display(
    resultados_diferencias.round(4)
)

,fold,ranking_jugador,ranking_rival,diff_ranking,edad_jugador,edad_rival,diff_edad,altura_jugador,altura_rival,diff_altura
0,1,0.2430,0.2430,0.4048,0.0173,0.0173,0.0296,0.0519,0.0519,0.0799
1,2,0.2429,0.2429,0.4023,0.0190,0.0190,0.0310,0.0508,0.0508,0.0777
2,3,0.2397,0.2397,0.3963,0.0191,0.0191,0.0307,0.0526,0.0526,0.0816
3,4,0.2368,0.2368,0.3898,0.0196,0.0196,0.0315,0.0515,0.0515,0.0794
4,5,0.2359,0.2359,0.3872,0.0130,0.0130,0.0224,0.0548,0.0548,0.0833


### Comparación temporal entre predictores conservados y descartados

Las comprobaciones anteriores han respaldado la estabilidad temporal de la redundancia entre ranking y puntos ATP y de la utilización de variables diferenciales jugador–rival. En el caso de las categorías de representación muy reducida, este notebook no vuelve a evaluar por fold su frecuencia individual, sino que comprueba posteriormente la trazabilidad de su exclusión respecto a las decisiones adoptadas en la selección original.

Como comprobación conjunta, se analiza ahora la evidencia supervisada correspondiente a los predictores que posteriormente formaron el modelo base y a aquellos candidatos que fueron descartados durante la depuración.

Esta comparación no pretende establecer que todos los predictores conservados deban presentar asociaciones marginales elevadas. Algunas variables contextuales pueden aportar información mediante relaciones no lineales o interacciones con otros predictores. El objetivo es detectar si las decisiones originales presentan algún comportamiento temporal claramente contradictorio con la evidencia disponible dentro de los conjuntos de entrenamiento.

In [9]:
predictores_finales_originales = [
    "tamano_cuadro",
    "numero_maximo_sets",
    "cabeza_serie",
    "cabeza_serie_rival",
    "ronda_encoded",
    "es_round_robin",
    "superficie_Clay",
    "superficie_Grass",
    "superficie_Hard",
    "nivel_torneo_ATP",
    "nivel_torneo_ATP_Finals",
    "nivel_torneo_Davis_Cup",
    "nivel_torneo_Grand_Slam",
    "nivel_torneo_Masters_1000",
    "mano_dominante_L",
    "mano_dominante_R",
    "tipo_entrada_DIRECT",
    "tipo_entrada_LL",
    "tipo_entrada_Q",
    "tipo_entrada_WC",
    "mano_dominante_rival_L",
    "mano_dominante_rival_R",
    "tipo_entrada_rival_DIRECT",
    "tipo_entrada_rival_LL",
    "tipo_entrada_rival_Q",
    "tipo_entrada_rival_WC"
]

# Las tres diferencias todavía no existían entre los 47 candidatos
# y se evaluaron aparte en el bloque anterior.
assert len(predictores_finales_originales) == 26

resultados_comparacion = []

for fold in sorted(resultados_supervisados["fold"].unique()):

    datos_fold = resultados_supervisados[
        resultados_supervisados["fold"].eq(fold)
    ].copy()

    datos_fold["grupo"] = np.where(
        datos_fold["variable"].isin(
            predictores_finales_originales
        ),
        "conservado",
        "no_conservado"
    )

    for grupo, datos_grupo in datos_fold.groupby("grupo"):

        resultados_comparacion.append(
            {
                "fold": fold,
                "grupo": grupo,
                "n_predictores": len(datos_grupo),
                "mediana_abs_spearman":
                    datos_grupo["spearman_porcentaje"]
                    .abs()
                    .median(),
                "mediana_mi":
                    datos_grupo["mi_porcentaje"]
                    .median()
            }
        )

resultados_comparacion = pd.DataFrame(
    resultados_comparacion
)

display(
    resultados_comparacion.round(4)
)

,fold,grupo,n_predictores,mediana_abs_spearman,mediana_mi
0,1,conservado,26,0.0113,0.0070
1,1,no_conservado,21,0.0173,0.0002
2,2,conservado,26,0.0104,0.0071
3,2,no_conservado,21,0.0190,0.0003
4,3,conservado,26,0.0130,0.0067
5,3,no_conservado,21,0.0191,0.0005
6,4,conservado,26,0.0126,0.0076
7,4,no_conservado,21,0.0196,0.0003
8,5,conservado,26,0.0126,0.0073
9,5,no_conservado,21,0.0130,0.0003


### Trazabilidad de los predictores descartados

La comparación agregada entre predictores conservados y descartados no se interpreta como un criterio de calidad de la selección, ya que las variables descartadas responden a motivos metodológicos diferentes.

En particular, algunas variables con asociación relevante fueron eliminadas por redundancia o sustituidas por representaciones diferenciales jugador–rival, mientras que otras correspondían a categorías de frecuencia extremadamente reducida.

Por este motivo, se comprueba que los 21 predictores candidatos no conservados puedan asignarse íntegramente a una de las razones de exclusión previamente definidas.

In [10]:
# ============================================================
# TRAZABILIDAD DE LOS 21 PREDICTORES NO CONSERVADOS
# ============================================================

eliminados_redundancia = [
    "puntos_ranking",
    "puntos_ranking_rival"
]

sustituidos_por_diferencias = [
    "ranking",
    "ranking_rival",
    "edad",
    "edad_rival",
    "altura",
    "altura_rival"
]

categorias_baja_representacion = [
    "superficie_Carpet",
    "superficie_UNKNOWN",
    "nivel_torneo_Olympics",
    "mano_dominante_UNKNOWN",
    "mano_dominante_rival_UNKNOWN",
    "tipo_entrada_ALT",
    "tipo_entrada_SE",
    "tipo_entrada_PR",
    "tipo_entrada_OTROS",
    "tipo_entrada_rival_ALT",
    "tipo_entrada_rival_SE",
    "tipo_entrada_rival_PR",
    "tipo_entrada_rival_OTROS"
]

no_conservados_esperados = set(
    eliminados_redundancia
    + sustituidos_por_diferencias
    + categorias_baja_representacion
)

no_conservados_reales = set(
    predictores_candidatos
) - set(
    predictores_finales_originales
)

assert no_conservados_reales == no_conservados_esperados

print("Predictores no conservados:", len(no_conservados_reales))
print("  Redundancia ranking-puntos:", len(eliminados_redundancia))
print("  Sustituidos por diferencias:", len(sustituidos_por_diferencias))
print("  Categorías de baja representación:", len(categorias_baja_representacion))
print("Todos los descartes tienen una justificación identificada.")

Predictores no conservados: 21
  Redundancia ranking-puntos: 2
  Sustituidos por diferencias: 6
  Categorías de baja representación: 13
Todos los descartes tienen una justificación identificada.


## 5. Auditoría temporal de las variables de carga competitiva

Una vez comprobada la estabilidad de las decisiones relativas a los predictores tradicionales, se analiza la selección de las variables de carga competitiva.

La selección original partió de distintas medidas de volumen, frecuencia, recencia y densidad competitiva. Posteriormente se redujo este conjunto considerando la redundancia entre medidas, su relación con el rendimiento, su interpretación deportiva y la utilidad de las diferencias jugador–rival.

La auditoría reproducirá únicamente las decisiones relevantes utilizando los conjuntos de entrenamiento de las mismas cinco particiones temporales. No se pretende realizar una nueva búsqueda de variables ni modificar las decisiones en función de los resultados.

La temporada 2024 permanece completamente excluida.

In [11]:
df_cargas_auditoria = pd.read_csv(
    "df_jugador_cargas_enriquecidas_final.csv",
    low_memory=False
)

df_cargas_auditoria["fecha_torneo"] = pd.to_datetime(
    df_cargas_auditoria["fecha_torneo"],
    errors="raise"
)

df_cargas_auditoria = (
    df_cargas_auditoria.loc[
        df_cargas_auditoria["es_desarrollo_modelo"].eq(1)
    ]
    .copy()
)

variables_carga_originales = [
    "juegos_365d",
    "sets_365d",
    "partidos_365d",
    "torneos_365d",

    "juegos_180d",
    "sets_180d",
    "partidos_180d",
    "torneos_180d",

    "juegos_temporada",
    "sets_temporada",
    "partidos_temporada",
    "torneos_temporada",

    "juegos_ultima_fecha_competitiva",
    "sets_ultima_fecha_competitiva",
    "partidos_ultima_fecha_competitiva",
    "torneos_ultima_fecha_competitiva",

    "dias_entre_inicios_competitivos",
    "semanas_con_competicion_56d"
]

columnas_necesarias = (
    variables_carga_originales
    + [
        "_id_partido",
        "jugador_id",
        "fecha_torneo",
        "match_year",
        "porcentaje_juegos_ganados",
        "historial_365d_completo",
        "historial_180d_completo",
        "historial_56d_completo",
        "historial_anterior_disponible"
    ]
)

faltantes = [
    columna
    for columna in columnas_necesarias
    if columna not in df_cargas_auditoria.columns
]

assert not faltantes, f"Faltan columnas: {faltantes}"

assert len(variables_carga_originales) == 18

assert 2024 not in df_cargas_auditoria["match_year"].unique()

assert (
    df_cargas_auditoria
    .groupby("_id_partido")
    .size()
    .eq(2)
    .all()
)

print("Observaciones:", len(df_cargas_auditoria))
print(
    "Partidos:",
    df_cargas_auditoria["_id_partido"].nunique()
)
print(
    "Variables originales de carga:",
    len(variables_carga_originales)
)
print(
    "Años:",
    sorted(df_cargas_auditoria["match_year"].unique())
)
print("Dataset de cargas para auditoría: CORRECTO")

Observaciones: 76292
Partidos: 38146
Variables originales de carga: 18
Años: [np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2022), np.int64(2023)]
Dataset de cargas para auditoría: CORRECTO


### Diferencia entre los conjuntos utilizados en ambos bloques

El conjunto previo a la selección de predictores tradicionales contiene 76.294 observaciones correspondientes a 38.147 partidos, mientras que el conjunto enriquecido con variables de carga contiene 76.292 observaciones y 38.146 partidos.

La diferencia corresponde a un único encuentro que fue eliminado posteriormente durante las comprobaciones de calidad por presentar cero juegos y cero sets registrados. Por tanto, esta diferencia responde al proceso de depuración previo a la construcción definitiva de las cargas y no a las particiones utilizadas en la auditoría.

### Estabilidad temporal de la redundancia entre medidas de volumen

La selección original de variables de carga mostró una elevada redundancia entre los juegos, los sets y los partidos disputados dentro de una misma ventana temporal.

Dado que estas tres medidas pretenden representar el volumen competitivo acumulado, se seleccionaron los juegos como medida representativa por ofrecer una mayor granularidad sobre la duración de la actividad competitiva.

Para comprobar que esta decisión no dependía del uso del conjunto completo de desarrollo, se recalculan las correlaciones de Spearman entre juegos, sets y partidos utilizando exclusivamente los datos de entrenamiento de cada fold temporal.

La comprobación se realiza para las ventanas de 365 días, 180 días, temporada y participación inmediatamente anterior. No se utiliza la variable objetivo, por lo que este análisis evalúa exclusivamente la estabilidad de la redundancia entre predictores.

In [12]:
familias_volumen = {
    "365d": [
        "juegos_365d",
        "sets_365d",
        "partidos_365d"
    ],
    "180d": [
        "juegos_180d",
        "sets_180d",
        "partidos_180d"
    ],
    "temporada": [
        "juegos_temporada",
        "sets_temporada",
        "partidos_temporada"
    ],
    "ultima_participacion": [
        "juegos_ultima_fecha_competitiva",
        "sets_ultima_fecha_competitiva",
        "partidos_ultima_fecha_competitiva"
    ]
}

resultados_redundancia_volumen = []

for fold in folds_auditoria:

    train = df_cargas_auditoria[
        df_cargas_auditoria["match_year"].isin(
            fold["anios_train"]
        )
    ]

    for familia, variables in familias_volumen.items():

        datos = train[variables].dropna()

        correlacion = datos.corr(
            method="spearman"
        )

        rho_juegos_sets = correlacion.iloc[0, 1]
        rho_juegos_partidos = correlacion.iloc[0, 2]
        rho_sets_partidos = correlacion.iloc[1, 2]

        resultados_redundancia_volumen.append(
            {
                "fold": fold["fold"],
                "familia": familia,
                "n_observaciones": len(datos),
                "rho_juegos_sets": rho_juegos_sets,
                "rho_juegos_partidos": rho_juegos_partidos,
                "rho_sets_partidos": rho_sets_partidos,
                "min_abs_rho": min(
                    abs(rho_juegos_sets),
                    abs(rho_juegos_partidos),
                    abs(rho_sets_partidos)
                )
            }
        )

resultados_redundancia_volumen = pd.DataFrame(
    resultados_redundancia_volumen
)

display(
    resultados_redundancia_volumen.round(4)
)

,fold,familia,n_observaciones,rho_juegos_sets,rho_juegos_partidos,rho_sets_partidos,min_abs_rho
0,1,365d,47442,0.9980,0.9932,0.9957,0.9932
1,1,180d,47442,0.9979,0.9904,0.9931,0.9904
2,1,temporada,47442,0.9993,0.9967,0.9975,0.9967
3,1,ultima_participacion,46483,0.9830,0.8911,0.9035,0.8911
4,2,365d,53226,0.9980,0.9932,0.9958,0.9932
5,2,180d,53226,0.9979,0.9905,0.9931,0.9905
6,2,temporada,53226,0.9993,0.9967,0.9974,0.9967
7,2,ultima_participacion,52179,0.9831,0.8914,0.9038,0.8914
8,3,365d,58990,0.9979,0.9931,0.9957,0.9931
9,3,180d,58990,0.9978,0.9902,0.9929,0.9902


### Comprobación específica de la carga de la participación anterior

A diferencia de las ventanas acumuladas de 365 días, 180 días y temporada, la redundancia entre juegos, sets y partidos de la participación inmediatamente anterior fue elevada pero no prácticamente perfecta.

En particular, los juegos y los sets mantuvieron una asociación muy alta, mientras que el número de partidos mostró una relación algo menor con ambas medidas.

Dado que la selección original conservó los juegos como representación de la carga inmediata, se comprueba adicionalmente si esta medida mantiene una asociación con `porcentaje_juegos_ganados` comparable o superior a la observada para sets y partidos dentro de cada conjunto de entrenamiento temporal.

Esta comprobación no pretende realizar una nueva selección, sino evaluar si la elección original de los juegos como medida más granular de volumen continúa estando respaldada temporalmente.

In [13]:
comparacion_ultima_participacion = []

for fold in folds_auditoria:

    train = df_cargas_auditoria[
        df_cargas_auditoria["match_year"].isin(
            fold["anios_train"]
        )
    ].copy()

    variables = [
        "juegos_ultima_fecha_competitiva",
        "sets_ultima_fecha_competitiva",
        "partidos_ultima_fecha_competitiva",
        "porcentaje_juegos_ganados"
    ]

    datos = train[variables].dropna()

    correlaciones = (
        datos
        .corr(method="spearman")[
            "porcentaje_juegos_ganados"
        ]
        .drop("porcentaje_juegos_ganados")
        .abs()
    )

    comparacion_ultima_participacion.append(
        {
            "fold": fold["fold"],
            "n_observaciones": len(datos),
            "juegos": correlaciones[
                "juegos_ultima_fecha_competitiva"
            ],
            "sets": correlaciones[
                "sets_ultima_fecha_competitiva"
            ],
            "partidos": correlaciones[
                "partidos_ultima_fecha_competitiva"
            ]
        }
    )

comparacion_ultima_participacion = pd.DataFrame(
    comparacion_ultima_participacion
)

display(
    comparacion_ultima_participacion.round(4)
)

,fold,n_observaciones,juegos,sets,partidos
0,1,46483,0.1534,0.1519,0.1553
1,2,52179,0.1507,0.1497,0.1541
2,3,57838,0.1479,0.1467,0.1509
3,4,63356,0.1470,0.1456,0.1493
4,5,68974,0.1458,0.1443,0.1480


#### Interpretación

Las tres medidas de la participación anterior presentan asociaciones de magnitud muy similar y mantienen un comportamiento estable entre folds. El número de partidos muestra una asociación marginal ligeramente superior a la de los juegos en las cinco particiones, mientras que los juegos superan ligeramente a los sets.

Esta diferencia no modifica la decisión original de utilizar los juegos como medida representativa. La selección no se basó exclusivamente en maximizar una correlación marginal, sino también en la fuerte redundancia existente entre las tres medidas y en la mayor granularidad de los juegos para representar participaciones de distinta duración.

Por tanto, la auditoría respalda la estabilidad de la familia de carga correspondiente a la participación anterior, aunque no demuestra una superioridad marginal de los juegos frente al número de partidos.

### Estabilidad temporal de las diferencias jugador–rival

La especificación principal de carga se construyó mediante diferencias entre la carga previa del jugador y la de su rival.

Esta representación se eligió porque permite expresar directamente la situación relativa de ambos participantes y, en el análisis original, mostró asociaciones más claras con el rendimiento que las cargas absolutas.

Para comprobar que esta decisión no dependía del uso del conjunto completo de desarrollo, se comparan dentro de cada conjunto de entrenamiento temporal las asociaciones de Spearman de las cargas absolutas y de sus correspondientes diferencias jugador–rival con `porcentaje_juegos_ganados`.

La comparación se realiza sobre muestras emparejadas para que ambas representaciones se evalúen utilizando exactamente las mismas observaciones.

In [14]:
variables_carga_comparar = [
    "juegos_365d",
    "juegos_temporada",
    "juegos_ultima_fecha_competitiva",
    "log_dias_entre_inicios_competitivos",
    "semanas_con_competicion_56d",
    "torneos_365d"
]

resultados_absoluto_diferencia = []

for fold in folds_auditoria:

    train = df_cargas_auditoria[
        df_cargas_auditoria["match_year"].isin(
            fold["anios_train"]
        )
    ].copy()

    for variable in variables_carga_comparar:

        variable_rival = f"{variable}_rival"
        variable_diff = f"diff_{variable}"

        columnas = [
            variable,
            variable_rival,
            variable_diff,
            "porcentaje_juegos_ganados"
        ]

        datos = train[columnas].dropna()

        corr = (
            datos
            .corr(method="spearman")[
                "porcentaje_juegos_ganados"
            ]
            .abs()
        )

        resultados_absoluto_diferencia.append(
            {
                "fold": fold["fold"],
                "variable": variable,
                "n_observaciones": len(datos),
                "absoluta_jugador": corr[variable],
                "absoluta_rival": corr[variable_rival],
                "diferencia": corr[variable_diff]
            }
        )

resultados_absoluto_diferencia = pd.DataFrame(
    resultados_absoluto_diferencia
)

display(
    resultados_absoluto_diferencia.round(4)
)

,fold,variable,n_observaciones,absoluta_jugador,absoluta_rival,diferencia
0,1,juegos_365d,47442,0.2395,0.2395,0.3975
1,1,juegos_temporada,47442,0.1365,0.1365,0.3325
2,1,juegos_ultima_fecha_competitiva,45642,0.1578,0.1578,0.2361
3,1,log_dias_entre_inicios_competitivos,45642,0.0536,0.0536,0.1327
4,1,semanas_con_competicion_56d,47442,0.0751,0.0751,0.1281
5,1,torneos_365d,47442,0.1166,0.1166,0.2046
6,2,juegos_365d,53226,0.2366,0.2366,0.3898
7,2,juegos_temporada,53226,0.1351,0.1351,0.3248
8,2,juegos_ultima_fecha_competitiva,51254,0.1548,0.1548,0.2307
9,2,log_dias_entre_inicios_competitivos,51254,0.0531,0.0531,0.1317


### Resultado de la representación relativa de la carga

Las diferencias jugador–rival presentaron asociaciones con `porcentaje_juegos_ganados` superiores a las correspondientes cargas absolutas en las seis dimensiones analizadas y en las cinco particiones temporales.

Este patrón se mantuvo al utilizar exclusivamente los datos de entrenamiento disponibles en cada fold, lo que indica que la mayor asociación observada para las variables relativas no depende de las temporadas posteriores del conjunto de desarrollo.

Los resultados respaldan la decisión de representar la carga competitiva principalmente mediante diferencias jugador–rival. Esta evidencia se interpreta como apoyo a la representación estadística de las variables y no como una demostración causal ni como una prueba aislada de mejora predictiva multivariante.

### Estabilidad temporal de la relación entre volumen y frecuencia competitiva

La variable `diff_torneos_365d` se reservó inicialmente para una especificación ampliada porque presentaba una dependencia relevante con `diff_juegos_365d`.

Ambas variables representan dimensiones relacionadas pero no idénticas: los juegos aproximan el volumen competitivo acumulado, mientras que los torneos representan principalmente la frecuencia de participación.

Para comprobar que la dependencia observada originalmente no estaba condicionada por el uso del conjunto completo de desarrollo, se recalcula su asociación dentro del entrenamiento de cada fold temporal.

In [15]:
dependencia_juegos_torneos = []

for fold in folds_auditoria:

    train = df_cargas_auditoria[
        df_cargas_auditoria["match_year"].isin(
            fold["anios_train"]
        )
    ].copy()

    datos = train[
        [
            "diff_juegos_365d",
            "diff_torneos_365d"
        ]
    ].dropna()

    rho = (
        datos
        .corr(method="spearman")
        .loc[
            "diff_juegos_365d",
            "diff_torneos_365d"
        ]
    )

    dependencia_juegos_torneos.append(
        {
            "fold": fold["fold"],
            "n_observaciones": len(datos),
            "rho_spearman": rho
        }
    )

dependencia_juegos_torneos = pd.DataFrame(
    dependencia_juegos_torneos
)

display(
    dependencia_juegos_torneos.round(4)
)

,fold,n_observaciones,rho_spearman
0,1,47442,0.7796
1,2,53226,0.7863
2,3,58990,0.7934
3,4,64560,0.7964
4,5,70360,0.8016


### Estabilidad temporal del control por avance de temporada

La carga acumulada durante la temporada depende parcialmente del momento del calendario en el que se disputa el encuentro, ya que la oportunidad de acumular actividad aumenta conforme avanza la temporada.

Para evaluar esta relación se utiliza una única observación por jugador y torneo. Esta unidad evita que una misma carga previa quede repetida tantas veces como partidos dispute un jugador dentro de una misma participación competitiva.

Dentro del conjunto de entrenamiento de cada fold se calcula la correlación de Spearman entre `dias_desde_inicio_temporada` y `juegos_temporada`.

Esta comprobación no utiliza la variable objetivo y permite evaluar si la justificación para incorporar `dias_desde_inicio_temporada` como variable de control permanece estable a lo largo de los distintos periodos de entrenamiento.

In [16]:
estabilidad_control_temporada_torneo = []

for fold in folds_auditoria:

    train = df_cargas_auditoria[
        df_cargas_auditoria["match_year"].isin(
            fold["anios_train"]
        )
    ].copy()

    # Una única observación por jugador y torneo.
    train_jugador_torneo = (
        train[
            [
                "jugador_id",
                "identificador_torneo",
                "dias_desde_inicio_temporada",
                "juegos_temporada"
            ]
        ]
        .drop_duplicates(
            subset=[
                "jugador_id",
                "identificador_torneo"
            ]
        )
        .dropna()
    )

    rho = (
        train_jugador_torneo[
            [
                "dias_desde_inicio_temporada",
                "juegos_temporada"
            ]
        ]
        .corr(method="spearman")
        .loc[
            "dias_desde_inicio_temporada",
            "juegos_temporada"
        ]
    )

    estabilidad_control_temporada_torneo.append(
        {
            "fold": fold["fold"],
            "n_jugador_torneo": len(
                train_jugador_torneo
            ),
            "rho_spearman": rho
        }
    )

estabilidad_control_temporada_torneo = pd.DataFrame(
    estabilidad_control_temporada_torneo
)

display(
    estabilidad_control_temporada_torneo.round(4)
)

,fold,n_jugador_torneo,rho_spearman
0,1,25256,0.6541
1,2,28315,0.6570
2,3,31374,0.6634
3,4,34331,0.6629
4,5,37413,0.6571


### Resultado del control temporal

La relación entre el avance de la temporada y los juegos acumulados presentó una elevada estabilidad temporal al utilizar una única observación por jugador y torneo. Las correlaciones de Spearman se situaron aproximadamente entre 0,65 y 0,66 en las cinco particiones de entrenamiento, reproduciendo además la magnitud observada en el análisis original.

Estos resultados confirman que la dependencia entre la carga estacional y el momento del calendario constituye una característica estructural y estable de los datos. Por tanto, se mantiene la incorporación de `dias_desde_inicio_temporada` como variable de control.

## 6. Conclusión de la auditoría temporal

La selección exploratoria original de variables se realizó sobre el conjunto completo de desarrollo. En consecuencia, aunque la temporada 2024 permaneció completamente excluida, las particiones internas utilizadas posteriormente durante el modelado no eran completamente independientes de todas las decisiones previas de selección.

La presente auditoría evaluó el impacto práctico de esta limitación reproduciendo las principales evidencias que justificaron la selección mediante los conjuntos de entrenamiento de las cinco particiones temporales.

Los resultados mostraron una elevada estabilidad temporal de la estructura de asociación de los predictores tradicionales, tanto respecto al desarrollo completo como al compararla con periodos posteriores no solapados. También se reprodujeron de forma consistente la elevada redundancia entre ranking y puntos ATP y la mayor asociación de las representaciones diferenciales jugador–rival. Para las categorías de representación muy reducida se verificó la trazabilidad de su exclusión respecto a los motivos establecidos durante la selección original, sin reinterpretarla como una nueva selección temporal por fold.

En las variables de carga competitiva también se mantuvieron las principales relaciones que justificaron su selección. La redundancia entre juegos, sets y partidos fue elevada y estable en las ventanas acumuladas; las diferencias jugador–rival presentaron asociaciones superiores a las cargas absolutas en todos los periodos analizados; la relación entre volumen y frecuencia competitiva permaneció estable; y la dependencia de la carga estacional respecto al avance del calendario se reprodujo utilizando la unidad jugador–torneo. En la participación inmediatamente anterior, juegos, sets y partidos mostraron asociaciones muy próximas, sin que los juegos fueran sistemáticamente la medida con mayor asociación marginal; su elección se mantiene por su mayor granularidad y por la elevada redundancia existente entre estas medidas.

Por tanto, la limitación metodológica asociada a la selección previa sobre el conjunto completo de desarrollo continúa existiendo formalmente. No obstante, esta auditoría no muestra evidencia de que haya modificado de forma sustancial las principales decisiones de selección adoptadas.

La temporada 2024 no ha participado en ninguna fase de esta auditoría y permanece reservada para la evaluación temporal externa.

### Resumen cuantitativo de la estabilidad temporal

Con el objetivo de sintetizar las principales comprobaciones de la auditoría temporal, se resumen cinco resultados directamente relacionados con las decisiones de selección adoptadas en el análisis exploratorio.

Para las medidas continuas se presentan la media, la desviación estándar muestral y el rango observado entre los cinco folds. La desviación estándar se utiliza únicamente como medida descriptiva de la variabilidad entre particiones, ya que los conjuntos de entrenamiento siguen una estructura expansiva y no constituyen muestras independientes.

En la comparación entre cargas absolutas y diferencias jugador–rival se resume, para cada fold, la proporción de las seis dimensiones analizadas en las que la asociación absoluta de la diferencia con el rendimiento supera a la correspondiente carga individual.

Este resumen no introduce nuevos criterios de selección ni modifica las decisiones previamente adoptadas; únicamente cuantifica su estabilidad temporal.

In [17]:
# ============================================================
# RESUMEN CUANTITATIVO DE LA AUDITORÍA TEMPORAL
# ============================================================


# ------------------------------------------------------------
# 1. Concordancia entrenamiento-validación
# ------------------------------------------------------------
# Correlación de rangos de Spearman entre la ordenación
# de las asociaciones predictor-rendimiento obtenidas en
# entrenamiento y en la temporada de validación posterior.

serie_train_valid = (
    estabilidad_fuera_muestra[
        "correlacion_rangos"
    ]
    .astype(float)
)


# ------------------------------------------------------------
# 2. Redundancia ranking-puntos ATP
# ------------------------------------------------------------
# Jugador y rival presentan los mismos valores debido a la
# estructura recíproca del conjunto. Se comprueba antes de
# resumir únicamente una de las dos perspectivas.

assert np.allclose(
    redundancia_ranking[
        "ranking_puntos_jugador"
    ],
    redundancia_ranking[
        "ranking_puntos_rival"
    ],
    equal_nan=True
)

serie_ranking_puntos = (
    redundancia_ranking[
        "ranking_puntos_jugador"
    ]
    .abs()
    .astype(float)
)


# ------------------------------------------------------------
# 3. Redundancia juegos-sets-partidos
# ------------------------------------------------------------
# Para cada fold se utiliza un criterio conservador:
# la MENOR correlación absoluta observada entre juegos,
# sets y partidos dentro de las tres ventanas acumuladas
# principales: 365 días, 180 días y temporada.
#
# Así, el resumen indica cuál fue incluso la asociación
# más débil dentro del bloque de volumen acumulado.

familias_acumuladas = [
    "365d",
    "180d",
    "temporada"
]

redundancia_acumulada = (
    resultados_redundancia_volumen[
        resultados_redundancia_volumen[
            "familia"
        ].isin(familias_acumuladas)
    ]
    .groupby(
        "fold",
        as_index=False
    )["min_abs_rho"]
    .min()
)

serie_redundancia_volumen = (
    redundancia_acumulada[
        "min_abs_rho"
    ]
    .astype(float)
)


# ------------------------------------------------------------
# 4. Diferencias jugador-rival frente a cargas absolutas
# ------------------------------------------------------------
# Para cada una de las seis dimensiones y cada fold se
# comprueba si:
#
# |rho(diferencia, rendimiento)|
# >
# max(
#     |rho(carga jugador, rendimiento)|,
#     |rho(carga rival, rendimiento)|
# )
#
# Posteriormente se calcula la proporción de comparaciones
# favorables dentro de cada fold.

comparacion_relativa = (
    resultados_absoluto_diferencia
    .copy()
)

comparacion_relativa[
    "diferencia_superior"
] = (
    comparacion_relativa[
        "diferencia"
    ]
    >
    comparacion_relativa[
        [
            "absoluta_jugador",
            "absoluta_rival"
        ]
    ].max(axis=1)
)

consistencia_diferencias = (
    comparacion_relativa
    .groupby("fold")[
        "diferencia_superior"
    ]
    .mean()
)

serie_diferencias = (
    consistencia_diferencias
    .astype(float)
)

n_comparaciones_diferencias = len(
    comparacion_relativa
)

n_favorables_diferencias = int(
    comparacion_relativa[
        "diferencia_superior"
    ].sum()
)


# ------------------------------------------------------------
# 5. Dependencia volumen-frecuencia
# ------------------------------------------------------------
# Asociación entre diff_juegos_365d y diff_torneos_365d.

serie_juegos_torneos = (
    dependencia_juegos_torneos[
        "rho_spearman"
    ]
    .abs()
    .astype(float)
)


# ------------------------------------------------------------
# FUNCIÓN DE RESUMEN
# ------------------------------------------------------------

def resumir_serie(nombre, serie):

    serie = pd.Series(
        serie,
        dtype=float
    ).dropna()

    assert len(serie) == 5

    return {
        "Comprobación": nombre,
        "Media": serie.mean(),
        "Desv. estándar": serie.std(ddof=1),
        "Mínimo": serie.min(),
        "Máximo": serie.max()
    }


# ------------------------------------------------------------
# TABLA RESUMEN
# ------------------------------------------------------------

resumen_auditoria = pd.DataFrame([
    resumir_serie(
        "Concordancia entrenamiento-validación",
        serie_train_valid
    ),

    resumir_serie(
        "Redundancia ranking-puntos ATP (|rho_s|)",
        serie_ranking_puntos
    ),

    resumir_serie(
        "Redundancia mínima del volumen acumulado (|rho_s|)",
        serie_redundancia_volumen
    ),

    resumir_serie(
        "Proporción de diferencias jugador-rival superiores",
        serie_diferencias
    ),

    resumir_serie(
        "Dependencia juegos-torneos 365d (|rho_s|)",
        serie_juegos_torneos
    )
])


# ------------------------------------------------------------
# RANGO PARA PRESENTACIÓN
# ------------------------------------------------------------

resumen_auditoria[
    "Rango"
] = (
    resumen_auditoria[
        "Mínimo"
    ].map(lambda x: f"{x:.4f}")
    +
    "–"
    +
    resumen_auditoria[
        "Máximo"
    ].map(lambda x: f"{x:.4f}")
)


# ------------------------------------------------------------
# TABLA FINAL DE REVISIÓN
# ------------------------------------------------------------

tabla_revision = (
    resumen_auditoria[
        [
            "Comprobación",
            "Media",
            "Desv. estándar",
            "Rango"
        ]
    ]
    .copy()
)

tabla_revision[
    "Media"
] = tabla_revision[
    "Media"
].round(4)

tabla_revision[
    "Desv. estándar"
] = tabla_revision[
    "Desv. estándar"
].round(4)


display(tabla_revision)


print(
    "\nComparaciones de cargas relativas favorables:",
    f"{n_favorables_diferencias}/{n_comparaciones_diferencias}"
)

assert n_comparaciones_diferencias == 30
assert n_favorables_diferencias == 30

,Comprobación,Media,Desv. estándar,Rango
0,Concordancia entrenamiento-validación,0.9771,0.0075,0.9641–0.9827
1,Redundancia ranking-puntos ATP (|rho_s|),0.9889,0.0008,0.9876–0.9896
2,Redundancia mínima del volumen acumulado (|rho...,0.9902,0.0002,0.9899–0.9905
3,Proporción de diferencias jugador-rival superi...,1.0000,0.0000,1.0000–1.0000
4,Dependencia juegos-torneos 365d (|rho_s|),0.7915,0.0086,0.7796–0.8016



Comparaciones de cargas relativas favorables: 30/30
